<a href="https://colab.research.google.com/github/chat-neha/airquality3.0/blob/LSTM/AirQuality3_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data prep

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.read_csv('data.csv')

# Convert 'datetime' column to datetime objects
df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')

print(df.head())

   Unnamed: 0            datetime        ws          wd       temp         rh  \
0           0 2011-01-01 05:30:00  2.748318  235.699036  11.052734  61.081372   
1           1 2011-01-01 08:30:00  3.285797  225.197205  12.509644  55.546595   
2           2 2011-01-01 11:30:00  3.731982  210.610474  22.320221  28.955012   
3           3 2011-01-01 14:30:00  3.831188  224.834351  25.073883  20.784734   
4           4 2011-01-01 17:30:00  2.461802  234.460114  23.678711  23.357070   

   dew_temp  precipitation     pressure        wv  ...  duaod550  omaod550  \
0  3.833771       0.000148  1007.853027  7.451631  ...  0.003228  0.086111   
1  3.849792       0.000000  1009.779663  7.143795  ...  0.002302  0.066265   
2  3.398590       0.000000  1010.135498  7.146959  ...  0.001736  0.053539   
3  1.071045       0.000000  1006.981384  7.514349  ...  0.001533  0.044161   
4  1.532501       0.000000  1006.195496  7.559806  ...  0.001438  0.063244   

   ssaod550  suaod550    aod469    aod550   

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# -----------------------------
# 1. Load & preprocess data
# -----------------------------
df = pd.read_csv("data.csv")

# Parse datetime and sort
df['datetime'] = pd.to_datetime(df['datetime'], format="%d-%m-%Y %H:%M")
df = df.sort_values("datetime")

# Extract only pm2p5 (you can extend later for multivariate)
data = df[['pm2p5']].values.astype(float)

# Scale (LSTMs work better with normalized data)
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# -----------------------------
# 2. Create dataset
# -----------------------------
WINDOW_SIZE = 24  # past 3 days (3-hourly data → 8 steps per day × 3 days)

class TimeSeriesDataset(Dataset):
    def __init__(self, series, window_size):
        self.series = series
        self.window_size = window_size

    def __len__(self):
        return len(self.series) - self.window_size

    def __getitem__(self, idx):
        x = self.series[idx:idx + self.window_size]
        y = self.series[idx + self.window_size]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

dataset = TimeSeriesDataset(data_scaled, WINDOW_SIZE)

# Train-test split (80-20)
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# -----------------------------
# 3. Define LSTM model
# -----------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # take last hidden state
        out = self.fc(out)
        return out

model = LSTMModel()

# -----------------------------
# 4. Training loop
# -----------------------------
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    for X, y in train_loader:
      # X already has shape (batch, window_size, 1)
      optimizer.zero_grad()
      output = model(X)   # no extra unsqueeze
      loss = criterion(output, y.unsqueeze(-1))
      loss.backward()
      optimizer.step()


    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item():.4f}")

# -----------------------------
# 5. Evaluation (example on test set)
# -----------------------------
# -----------------------------
# 5. Evaluation (example on test set)
# -----------------------------
model.eval()
preds, actuals = [], []
with torch.no_grad():
    for X, y in test_loader:
        output = model(X)
        preds.append(output.numpy())
        actuals.append(y.numpy())

# Flatten properly to 2D
preds = np.concatenate(preds).reshape(-1, 1)
actuals = np.concatenate(actuals).reshape(-1, 1)

# Inverse scale back to original pm2p5 values
preds_inv = scaler.inverse_transform(preds)
actuals_inv = scaler.inverse_transform(actuals)

print("Sample predictions vs actuals:")
print(np.hstack([preds_inv[:5], actuals_inv[:5]]))


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:616: UserWarning: Using a target size (torch.Size([64, 1, 1])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:616: UserWarning: Using a target size (torch.Size([24, 1, 1])) that is different to the input size (torch.Size([24, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/20, Loss: 0.0182
Epoch 2/20, Loss: 0.0490
Epoch 3/20, Loss: 0.0088
Epoch 4/20, Loss: 0.0197
Epoch 5/20, Loss: 0.0215
Epoch 6/20, Loss: 0.0320
Epoch 7/20, Loss: 0.0252
Epoch 8/20, Loss: 0.0364
Epoch 9/20, Loss: 0.0280
Epoch 10/20, Loss: 0.0279
Epoch 11/20, Loss: 0.0413
Epoch 12/20, Loss: 0.0272
Epoch 13/20, Loss: 0.0323
Epoch 14/20, Loss: 0.0127
Epoch 15/20, Loss: 0.0141
Epoch 16/20, Loss: 0.0258
Epoch 17/20, Loss: 0.0203
Epoch 18/20, Loss: 0.0349
Epoch 19/20, Loss: 0.0184
Epoch 20/20, Loss: 0.0277
Sample predictions vs actuals:
[[ 80.39113   63.66314 ]
 [ 80.84937  138.04681 ]
 [ 80.31211   51.05443 ]
 [ 81.78046  182.47183 ]
 [ 80.44147   45.052864]]


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# -----------------------------
# 1. Load & preprocess data
# -----------------------------
df = pd.read_csv("data.csv")
df['datetime'] = pd.to_datetime(df['datetime'], format="%d-%m-%Y %H:%M")
df = df.sort_values("datetime").reset_index(drop=True)

timestamps = df['datetime'].values
data = df[['pm2p5']].values.astype(float)

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# -----------------------------
# 2. Create dataset for multi-step
# -----------------------------
WINDOW_SIZE = 24  # past 3 days (3-hourly)
FORECAST_HORIZON = 8  # next 24 hours (8x3h)

class MultiStepDataset(Dataset):
    def __init__(self, series, window_size, horizon):
        self.series = series
        self.window_size = window_size
        self.horizon = horizon

    def __len__(self):
        return len(self.series) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        x = self.series[idx:idx+self.window_size]
        y = self.series[idx+self.window_size: idx+self.window_size+self.horizon]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

dataset = MultiStepDataset(data_scaled, WINDOW_SIZE, FORECAST_HORIZON)

# Train-test split (80-20)
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# -----------------------------
# 3. Define LSTM model (multi-step)
# -----------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=FORECAST_HORIZON):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # take last hidden state
        out = self.fc(out)
        return out

model = LSTMModel()

# -----------------------------
# 4. Training loop
# -----------------------------
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        # X is already (batch, window_size, 1)
        optimizer.zero_grad()
        output = model(X)  # (batch, horizon)
        loss = criterion(output, y.squeeze(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {total_loss/len(train_loader):.4f}")
# -----------------------------
# 5. Evaluation
# -----------------------------
def evaluate(loader):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for X, y in loader:
            output = model(X)  # (batch, horizon)
            preds.append(output.numpy())
            actuals.append(y.squeeze(-1).numpy())
    preds = np.concatenate(preds)
    actuals = np.concatenate(actuals)
    preds_inv = scaler.inverse_transform(preds)
    actuals_inv = scaler.inverse_transform(actuals)
    return preds_inv, actuals_inv


train_preds, train_actuals = evaluate(train_loader)
test_preds, test_actuals = evaluate(test_loader)

# -----------------------------
# 6. Metrics
# -----------------------------
def compute_metrics(y_true, y_pred, name=""):
    mse = mean_squared_error(y_true.flatten(), y_pred.flatten())
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true.flatten(), y_pred.flatten())
    r2 = r2_score(y_true.flatten(), y_pred.flatten())
    print(f"{name} - R2: {r2:.4f}, MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}")

compute_metrics(train_actuals, train_preds, "Train")
compute_metrics(test_actuals, test_preds, "Test")

# -----------------------------
# 7. Show sample predictions with datetime
# -----------------------------
# Align timestamps with test predictions
start_idx = train_size + WINDOW_SIZE
pred_timestamps = df['datetime'].iloc[start_idx : start_idx + len(test_preds)]

for i in range(3):  # show first 3 samples
    print(f"\nPrediction window {i+1} starting at {pred_timestamps.iloc[i]}")
    for step in range(FORECAST_HORIZON):
        t = pred_timestamps.iloc[i] + pd.Timedelta(hours=3*(step+1))
        print(f"{t} | Pred: {test_preds[i, step]:.2f}, Actual: {test_actuals[i, step]:.2f}")


Epoch 1/20, Train Loss: 0.0186
Epoch 2/20, Train Loss: 0.0080
Epoch 3/20, Train Loss: 0.0070
Epoch 4/20, Train Loss: 0.0067
Epoch 5/20, Train Loss: 0.0064
Epoch 6/20, Train Loss: 0.0063
Epoch 7/20, Train Loss: 0.0061
Epoch 8/20, Train Loss: 0.0060
Epoch 9/20, Train Loss: 0.0059
Epoch 10/20, Train Loss: 0.0059
Epoch 11/20, Train Loss: 0.0058
Epoch 12/20, Train Loss: 0.0058
Epoch 13/20, Train Loss: 0.0058
Epoch 14/20, Train Loss: 0.0057
Epoch 15/20, Train Loss: 0.0057
Epoch 16/20, Train Loss: 0.0057
Epoch 17/20, Train Loss: 0.0056
Epoch 18/20, Train Loss: 0.0057
Epoch 19/20, Train Loss: 0.0056
Epoch 20/20, Train Loss: 0.0056
Train - R2: 0.7910, MAE: 15.2603, MSE: 456.9185, RMSE: 21.3757
Test - R2: 0.7808, MAE: 15.4133, MSE: 475.0780, RMSE: 21.7963

Prediction window 1 starting at 2018-03-14 14:30:00
2018-03-14 17:30:00 | Pred: 93.70, Actual: 62.30
2018-03-14 20:30:00 | Pred: 99.36, Actual: 70.27
2018-03-14 23:30:00 | Pred: 69.21, Actual: 38.15
2018-03-15 02:30:00 | Pred: 48.99, Actual: 2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ============================================================
# 1. Load & preprocess data
# ============================================================
df = pd.read_csv("data.csv")
df['datetime'] = pd.to_datetime(df['datetime'], format="%d-%m-%Y %H:%M")
df = df.sort_values("datetime").reset_index(drop=True)

timestamps = df['datetime'].values
data = df[['pm2p5']].values.astype(float)

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# ============================================================
# 2. Create dataset for multi-step forecasting
# ============================================================
WINDOW_SIZE = 24      # past 3 days (3-hourly)
FORECAST_HORIZON = 8  # next 24 hours (8x3h)

class MultiStepDataset(Dataset):
    def __init__(self, series, window_size, horizon):
        self.series = series
        self.window_size = window_size
        self.horizon = horizon

    def __len__(self):
        return len(self.series) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        x = self.series[idx:idx+self.window_size]
        y = self.series[idx+self.window_size: idx+self.window_size+self.horizon]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

dataset = MultiStepDataset(data_scaled, WINDOW_SIZE, FORECAST_HORIZON)

# Train-test split (80-20, keeping order since it's time series)
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

# ============================================================
# 3. Define LSTM model
# ============================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=FORECAST_HORIZON):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # take last hidden state
        out = self.fc(out)
        return out

# ============================================================
# 4. Metrics function
# ============================================================
def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true.flatten(), y_pred.flatten())
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true.flatten(), y_pred.flatten())
    r2 = r2_score(y_true.flatten(), y_pred.flatten())
    return {"R2": r2, "MAE": mae, "MSE": mse, "RMSE": rmse}

# ============================================================
# 5. Training & evaluation function
# ============================================================
def train_and_evaluate(hidden_size, num_layers, lr, batch_size, epochs=10):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = LSTMModel(hidden_size=hidden_size, num_layers=num_layers)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X, y in train_loader:
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output, y.squeeze(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # Evaluation
    def evaluate(loader):
        model.eval()
        preds, actuals = [], []
        with torch.no_grad():
            for X, y in loader:
                output = model(X)
                preds.append(output.numpy())
                actuals.append(y.squeeze(-1).numpy())
        preds = np.concatenate(preds)
        actuals = np.concatenate(actuals)
        preds_inv = scaler.inverse_transform(preds)
        actuals_inv = scaler.inverse_transform(actuals)
        return preds_inv, actuals_inv

    train_preds, train_actuals = evaluate(train_loader)
    test_preds, test_actuals = evaluate(test_loader)

    train_metrics = compute_metrics(train_actuals, train_preds)
    test_metrics = compute_metrics(test_actuals, test_preds)

    return train_metrics, test_metrics

# ============================================================
# 6. Grid search
# ============================================================
hidden_sizes = [32, 64]
num_layers_list = [1, 2]
lrs = [0.001, 0.005]
batch_sizes = [32, 64]

results = []

for hs in hidden_sizes:
    for nl in num_layers_list:
        for lr in lrs:
            for bs in batch_sizes:
                train_metrics, test_metrics = train_and_evaluate(hs, nl, lr, bs, epochs=15)
                results.append({
                    "hidden_size": hs,
                    "num_layers": nl,
                    "lr": lr,
                    "batch_size": bs,
                    "train_R2": train_metrics["R2"],
                    "train_MAE": train_metrics["MAE"],
                    "train_MSE": train_metrics["MSE"],
                    "train_RMSE": train_metrics["RMSE"],
                    "test_R2": test_metrics["R2"],
                    "test_MAE": test_metrics["MAE"],
                    "test_MSE": test_metrics["MSE"],
                    "test_RMSE": test_metrics["RMSE"],
                })
                print(f"✅ hs={hs}, nl={nl}, lr={lr}, bs={bs} -> Test R2={test_metrics['R2']:.4f}")

# Put into DataFrame
results_df = pd.DataFrame(results)
print("\n===== Grid Search Results =====")
print(results_df)


✅ hs=32, nl=1, lr=0.001, bs=32 -> Test R2=0.7763
✅ hs=32, nl=1, lr=0.001, bs=64 -> Test R2=0.7687
✅ hs=32, nl=1, lr=0.005, bs=32 -> Test R2=0.7825
✅ hs=32, nl=1, lr=0.005, bs=64 -> Test R2=0.7752
✅ hs=32, nl=2, lr=0.001, bs=32 -> Test R2=0.7837
✅ hs=32, nl=2, lr=0.001, bs=64 -> Test R2=0.7680
✅ hs=32, nl=2, lr=0.005, bs=32 -> Test R2=0.7901
✅ hs=32, nl=2, lr=0.005, bs=64 -> Test R2=0.7849
✅ hs=64, nl=1, lr=0.001, bs=32 -> Test R2=0.7802
✅ hs=64, nl=1, lr=0.001, bs=64 -> Test R2=0.7761
✅ hs=64, nl=1, lr=0.005, bs=32 -> Test R2=0.7883
✅ hs=64, nl=1, lr=0.005, bs=64 -> Test R2=0.7850
✅ hs=64, nl=2, lr=0.001, bs=32 -> Test R2=0.7823
✅ hs=64, nl=2, lr=0.001, bs=64 -> Test R2=0.7817
✅ hs=64, nl=2, lr=0.005, bs=32 -> Test R2=0.7951
✅ hs=64, nl=2, lr=0.005, bs=64 -> Test R2=0.7791

===== Grid Search Results =====
    hidden_size  num_layers     lr  batch_size  train_R2  train_MAE  \
0            32           1  0.001          32  0.784383  15.359621   
1            32           1  0.001       

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from itertools import product

# -----------------------------
# 1. Load & preprocess data
# -----------------------------
df = pd.read_csv("data.csv")
df['datetime'] = pd.to_datetime(df['datetime'], format="%d-%m-%Y %H:%M")
df = df.sort_values("datetime").reset_index(drop=True)

timestamps = df['datetime'].values
data = df.drop(columns=['datetime']).values.astype(float)  # all features

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# -----------------------------
# 2. Dataset
# -----------------------------
WINDOW_SIZE = 24   # past 3 days (3-hourly)
FORECAST_HORIZON = 8  # next 24h (8 x 3h)

class MultiStepDataset(Dataset):
    def __init__(self, series, window_size, horizon, target_col=0):
        self.series = series
        self.window_size = window_size
        self.horizon = horizon
        self.target_col = target_col

    def __len__(self):
        return len(self.series) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        x = self.series[idx:idx+self.window_size]         # (window, features)
        y = self.series[idx+self.window_size: idx+self.window_size+self.horizon, self.target_col]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

dataset = MultiStepDataset(data_scaled, WINDOW_SIZE, FORECAST_HORIZON, target_col=0)

# Train-test split (80-20)
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

# -----------------------------
# 3. Model
# -----------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=FORECAST_HORIZON):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)       # (batch, window, hidden)
        out = out[:, -1, :]         # last hidden state
        out = self.fc(out)          # (batch, horizon)
        return out

# -----------------------------
# 4. Training + evaluation
# -----------------------------
def train_model(model, train_loader, val_loader, epochs, lr):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X, y in train_loader:
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

def evaluate(model, loader):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for X, y in loader:
            output = model(X)
            preds.append(output.numpy())
            actuals.append(y.numpy())
    preds = np.concatenate(preds)
    actuals = np.concatenate(actuals)
    # inverse scale only target (first column)
    pm2p5_scaler = MinMaxScaler()
    pm2p5_scaler.min_, pm2p5_scaler.scale_ = scaler.min_[0], scaler.scale_[0]
    preds_inv = pm2p5_scaler.inverse_transform(preds)
    actuals_inv = pm2p5_scaler.inverse_transform(actuals)
    return preds_inv, actuals_inv

def compute_metrics(y_true, y_pred, name=""):
    mse = mean_squared_error(y_true.flatten(), y_pred.flatten())
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true.flatten(), y_pred.flatten())
    r2 = r2_score(y_true.flatten(), y_pred.flatten())
    print(f"{name} - R2: {r2:.4f}, MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}")
    return r2

# -----------------------------
# 5. Grid Search
# -----------------------------
param_grid = {
    "hidden_size": [32, 64, 128],
    "num_layers": [1, 2],
    "lr": [0.001, 0.0005]
}
best_score = -np.inf
best_params = None

for hidden_size, num_layers, lr in product(param_grid["hidden_size"], param_grid["num_layers"], param_grid["lr"]):
    print(f"\nTesting hidden_size={hidden_size}, num_layers={num_layers}, lr={lr}")
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    model = LSTMModel(input_size=data.shape[1], hidden_size=hidden_size, num_layers=num_layers)
    train_model(model, train_loader, test_loader, epochs=20, lr=lr)

    train_preds, train_actuals = evaluate(model, train_loader)
    test_preds, test_actuals = evaluate(model, test_loader)

    r2 = compute_metrics(test_actuals, test_preds, "Test")
    if r2 > best_score:
        best_score = r2
        best_params = (hidden_size, num_layers, lr)

print(f"\nBest R²: {best_score:.4f} with params hidden_size={best_params[0]}, num_layers={best_params[1]}, lr={best_params[2]}")



Testing hidden_size=32, num_layers=1, lr=0.001
Epoch 5/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0000
Epoch 15/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000
Test - R2: 0.9999, MAE: 71.6156, MSE: 7925.8022, RMSE: 89.0270

Testing hidden_size=32, num_layers=1, lr=0.0005
Epoch 5/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0000
Epoch 15/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000
Test - R2: 0.9999, MAE: 54.6022, MSE: 4771.7349, RMSE: 69.0777

Testing hidden_size=32, num_layers=2, lr=0.001
Epoch 5/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0000
Epoch 15/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000
Test - R2: 0.9999, MAE: 58.6069, MSE: 5005.7471, RMSE: 70.7513

Testing hidden_size=32, num_layers=2, lr=0.0005
Epoch 5/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0000
Epoch 15/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000
Test - R2: 0.9999, MAE: 62.4004, MSE: 6320.2573, RMSE: 79.5000

Testing hidden_size=64, num_layers=1, lr=0.001
Epoch 5/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0000
Epoch 15/20, Loss: 0.0000
Epoch 20/20, Loss